In [ ]:
import torch
from torch import nn
from torch.optim import Adam
from torchvision.transforms import transforms
from torch.utils.data import DataLoader,Dataset
from torchvision import models
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import numpy as np




In [ ]:
# if(torch.mps.is_available):
#     print("MPS available")

device="mps" if torch.mps.is_available() else "cpu"   

In [ ]:
device

In [ ]:
import pandas as pd
from pathlib import Path

base_path = Path("/Users/ritabratadas/Desktop/STUDY FOLDERS/Machine Learning/DL_datasets/bean-leaf")

train_df = pd.read_csv(base_path / "train.csv")
val_df = pd.read_csv(base_path / "val.csv")

for df in (train_df, val_df):
    # val.csv has padded headers/values, so normalize both files the same way.
    df.columns = df.columns.str.strip()
    df["image:FILE"] = df["image:FILE"].astype(str).str.strip()

# create full path
train_df["image:FILE"] = train_df["image:FILE"].map(lambda p: str(base_path / p))
val_df["image:FILE"] = val_df["image:FILE"].map(lambda p: str(base_path / p))

In [ ]:
train_df.head()

In [ ]:
train_df.columns = train_df.columns.str.strip()
train_df["category"].value_counts()

In [ ]:
print(train_df.shape)
print(val_df.shape)

In [ ]:
transform=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float)

])

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self,dataframe,transform):
        self.dataframe=dataframe
        self.transform=transform
        self.labels=torch.tensor(dataframe["category"]).to(device)
    def __len__(self):
        return self.dataframe.shape[0]
    def __getitem__(self, index):
        img_path=self.dataframe.iloc[index,0]
        label=self.labels[index]
        image=Image.open(img_path)
        if self.transform:
            image=(self.transform(image)/255.0).to(device)
            return image,label        
         

In [ ]:
train_dataset=CustomImageDataset(dataframe=train_df,transform=transform)
val_dataset=CustomImageDataset(dataframe=val_df,transform=transform)

In [ ]:
n_rows=3
n_col=3

f, axarr=plt.subplots(n_rows,n_col)

for row in range(n_rows):
    for cols in range(n_col):
        image=train_dataset[np.random.randint(0,train_dataset.__len__())][0].cpu()
        axarr[row,cols].imshow((image*255.0).squeeze().permute(1,2,0))
        axarr[row,cols].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
LR=1e-3
BATCH_SIZE=4
EPOCHS=15

In [ ]:
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [ ]:
googlenet_model=models.googlenet(weights="DEFAULT")

In [ ]:
for param in googlenet_model.parameters():
    param.requires_grad=True
    

In [ ]:
googlenet_model.fc

In [ ]:
num_classes=len(train_df["category"].unique())
num_classes

In [ ]:
googlenet_model.fc=torch.nn.Linear(googlenet_model.fc.in_features, num_classes)
googlenet_model.fc

In [ ]:
googlenet_model.to(device)

In [ ]:
from tqdm.auto import tqdm
loss_fun=nn.CrossEntropyLoss()
optimizer=Adam(googlenet_model.parameters(),lr=LR)

total_loss_train_plot=[]
total_acc_train_plot=[]

for epoch in tqdm(range(EPOCHS)):
    total_acc_train=0
    total_loss_train=0

    for inputs,labels in tqdm(train_loader):
        optimizer.zero_grad()
        outputs=googlenet_model(inputs)
        train_loss=loss_fun(outputs,labels)
        total_loss_train+=train_loss.item()

        train_loss.backward()

        train_acc=(torch.argmax(outputs,axis=1)==labels).sum().item()
        total_acc_train+=train_acc
        optimizer.step()
    total_loss_train_plot.append(round(total_loss_train/1000,4))
    total_acc_train_plot.append(round(total_acc_train/train_dataset.__len__()*100,4))
    print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {round(total_loss_train/1000,4)}, Train Accuracy: {round(total_acc_train/train_dataset.__len__()*100,4)} %")

In [ ]:
import torch

torch.save(googlenet_model.state_dict(), "bean_leaf_googlenet.pth")